# einops-reduce-min — ex1: per-channel min floor for a feature map

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-reduce-min`. Running the final beacon cell reports progress against the `Einops: Reduce with min` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce with min` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-reduce-min`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce-min"
DD_SUBTOPIC = "Einops: Reduce with min"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.reduce with 'min' — quick refresher

`reduce(x, '<in> -> <out>', 'min')` collapses the axes dropped on the right side, taking the elementwise **minimum** over each collapsed group. Same syntax as `'mean'` / `'sum'` / `'max'` — only the op changes.

Common uses: per-channel **min** for a channel-floor normalization, per-row **min** to identify the worst feature in a batch, or windowed-min for a max-pool-style erosion. Composable with `(...)` decomposition: `reduce(x, 'b (h h2) (w w2) -> b h w', 'min', h2=2, w2=2)` is a 2×2 min-pool.

**Min vs max symmetry.** Every property of `'max'` reductions has a twin: `min(-x) == -max(x)`, min-pooling is dilation's dual, and argmin gates the same flow patterns as argmax.

### Exercise 1 — per-channel min floor for a feature map

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `einops.reduce(..., 'min')` to compute a per-channel spatial minimum across `(B, C, H, W)`, returning the resulting `(B, C)` floor that can be subtracted to zero-floor each map.
> Keywords: min, reduce, channel-floor, broadcasting
> ```

**KCs targeted:** `reduce-min-op`, `reduce-axis-collapse`

Implement `ex1_channel_min_floor(x)`. Given an activation tensor shaped `(B, C, H, W)`:

1. Use `einops.reduce` with the `'min'` op to collapse the spatial axes `H` and `W`, leaving `(B, C)`.
2. The result is the per-sample, per-channel **floor** — the smallest value across the spatial map.

Input: `(B, C, H, W)` float tensor.
Output: `(B, C)` float tensor containing the spatial-min of each channel.

You must use `einops.reduce` (NOT `x.min()` / `x.amin()`).

In [ ]:
def ex1_channel_min_floor(x: Tensor) -> Tensor:
    return reduce(x, 'b c h w -> b c', 'min')


<details><summary>Solution</summary>

```python
def ex1_channel_min_floor(x: Tensor) -> Tensor:
    return reduce(x, 'b c h w -> b c', 'min')
```

**The pattern.** `'b c h w -> b c'` drops `h` and `w`, so `min` is taken across all H*W spatial positions of each (batch, channel) pair. This is the same shape contract as `x.amin(dim=(-2, -1))` but reads more declaratively — you can see which axes survive.

**Use case — floor subtraction.** `floor = ex1_channel_min_floor(x)`; `normalized = x - floor[..., None, None]` zero-floors each channel of each sample so the smallest value becomes 0. This is how some attention-vis pipelines stabilize displays.

**Why not `x.amin(dim=(-2, -1))`.** Both work. `einops.reduce` wins when the pattern is part of a larger pipeline whose other steps are already `rearrange` / `repeat` — keeping a uniform vocabulary makes the code readable.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()